# Spatial Analysis of Zillow Home Sales

This notebook explores the spatial distribution of Zillow home sales using geocoded metropolitan markets and H3 spatial indexing.

The goal is to identify geographic concentration patterns in sales activity and prepare spatial features that can support downstream forecasting experiments.

## Libraries

In [1]:
import os
import time
import h3
import numpy as np
import pandas as pd
import folium
import branca.colormap as cm
import geopandas as gpd

from geopy.geocoders import ArcGIS, Nominatim
from shapely.geometry import Polygon

## Load hierarchical sales data

The analysis starts from the cleaned hierarchical Zillow dataset. Only metropolitan-level series are used for the spatial analysis because they represent the most granular geographic markets in the hierarchy.

In [2]:
df = pd.read_csv("Data/zillow_houses_hierarchy_cleaned.csv", parse_dates=["ds"])

df["y"] = pd.to_numeric(df["y"], errors="coerce").fillna(0)
df["hierarchy_level"] = df["hierarchy_level"].astype(int)

display(df.head())

,unique_id,ds,y,RegionName,StateName,hierarchy_level
0,US,2008-02-29,174474.0,United States,NaN,0
1,US,2008-03-31,203564.0,United States,NaN,0
2,US,2008-04-30,225601.0,United States,NaN,0
3,US,2008-05-31,248297.0,United States,NaN,0
4,US,2008-06-30,260848.0,United States,NaN,0


## Extract metropolitan markets

Unique metropolitan markets are extracted to create a geocoding lookup table. Each market is represented by a single latitude and longitude, which provides a centroid-based approximation of its location.

In [3]:
regions = (
    df[df["hierarchy_level"] == 2]
    [["unique_id", "RegionName", "StateName"]]
    .drop_duplicates()
    .sort_values("unique_id")
    .reset_index(drop=True)
)

regions["query"] = regions["RegionName"] + ", USA"

print("Unique metropolitan markets:", len(regions))
display(regions.head())

Unique metropolitan markets: 300


,unique_id,RegionName,StateName,query
0,US|AK|394327,"Anchorage, AK",AK,"Anchorage, AK, USA"
1,US|AL|394351,"Auburn, AL",AL,"Auburn, AL, USA"
2,US|AL|394388,"Birmingham, AL",AL,"Birmingham, AL, USA"
3,US|AL|394519,"Daphne, AL",AL,"Daphne, AL, USA"
4,US|AL|394537,"Dothan, AL",AL,"Dothan, AL, USA"


## Geocoding with cache

Market names are geocoded once and saved locally

In [4]:
geo_path = "Data/zillow_region_geocodes.csv"

os.makedirs(os.path.dirname(geo_path), exist_ok=True)

geo_cols = ["unique_id", "RegionName", "StateName", "query", "lat", "lon", "source"]

arcgis = ArcGIS(timeout=5)
nominatim = Nominatim(user_agent="zillow_housing_spatial")


def geocode_market(query):
    try:
        loc = arcgis.geocode(query, exactly_one=True)
        if loc is not None:
            return loc.latitude, loc.longitude, "ArcGIS"
    except Exception:
        pass

    try:
        loc = nominatim.geocode(
            query,
            exactly_one=True,
            timeout=5,
            country_codes="us"
        )
        if loc is not None:
            return loc.latitude, loc.longitude, "Nominatim"
    except Exception:
        pass

    return np.nan, np.nan, None

In [5]:
if os.path.exists(geo_path):
    region_geo = pd.read_csv(geo_path)
else:
    region_geo = pd.DataFrame(columns=geo_cols)

region_geo = region_geo.reindex(columns=geo_cols)
region_geo = region_geo[region_geo["unique_id"].isin(regions["unique_id"])].copy()

done_ids = set(region_geo.dropna(subset=["lat", "lon"])["unique_id"])
pending = regions[~regions["unique_id"].isin(done_ids)].copy()

print("Already geocoded:", len(done_ids))
print("Pending:", len(pending))

new_rows = []

for _, row in pending.iterrows():
    lat, lon, source = geocode_market(row["query"])

    new_rows.append({
        "unique_id": row["unique_id"],
        "RegionName": row["RegionName"],
        "StateName": row["StateName"],
        "query": row["query"],
        "lat": lat,
        "lon": lon,
        "source": source
    })

    if len(new_rows) % 25 == 0:
        tmp = pd.concat([region_geo, pd.DataFrame(new_rows)], ignore_index=True)
        tmp = tmp.drop_duplicates("unique_id", keep="last")
        tmp.to_csv(geo_path, index=False)
        print(f"Saved {len(tmp)} geocodes...")

    time.sleep(1.0 if source == "Nominatim" else 0.1)

if new_rows:
    region_geo = pd.concat([region_geo, pd.DataFrame(new_rows)], ignore_index=True)

region_geo = (
    region_geo
    .drop_duplicates("unique_id", keep="last")
    .sort_values("unique_id")
    .reset_index(drop=True)
)

region_geo.to_csv(geo_path, index=False)

print("Final geocodes:", len(region_geo))
print("Missing geocodes:", region_geo[["lat", "lon"]].isna().any(axis=1).sum())

display(region_geo.head())

Already geocoded: 300
Pending: 0
Final geocodes: 300
Missing geocodes: 0


,unique_id,RegionName,StateName,query,lat,lon,source
0,US|AK|394327,"Anchorage, AK",AK,"Anchorage, AK, AK, USA",61.216583,-149.899597,ArcGIS
1,US|AL|394351,"Auburn, AL",AL,"Auburn, AL, AL, USA",32.604091,-85.512268,ArcGIS
2,US|AL|394388,"Birmingham, AL",AL,"Birmingham, AL, AL, USA",33.492037,-86.864467,ArcGIS
3,US|AL|394519,"Daphne, AL",AL,"Daphne, AL, AL, USA",30.869567,-87.776489,ArcGIS
4,US|AL|394537,"Dothan, AL",AL,"Dothan, AL, AL, USA",31.239375,-85.405742,ArcGIS


In [6]:
# Inspect missing geocodes
missing_geo = region_geo[region_geo["lat"].isna() | region_geo["lon"].isna()].copy()

print("Missing geocodes:", len(missing_geo))
display(missing_geo)

Missing geocodes: 0


,unique_id,RegionName,StateName,query,lat,lon,source


## H3 spatial indexing

Each geocoded market is assigned to an H3 cell. This converts point locations into spatial units that can be aggregated and compared across the country.

In [7]:
# Assign H3 cells
h3_res = 4

region_geo["h3_cell"] = region_geo.apply(
    lambda row: h3.latlng_to_cell(row["lat"], row["lon"], h3_res)
    if pd.notna(row["lat"]) and pd.notna(row["lon"])
    else None,
    axis=1
)

display(region_geo.head())

,unique_id,RegionName,StateName,query,lat,lon,source,h3_cell
0,US|AK|394327,"Anchorage, AK",AK,"Anchorage, AK, AK, USA",61.216583,-149.899597,ArcGIS,840c733ffffffff
1,US|AL|394351,"Auburn, AL",AL,"Auburn, AL, AL, USA",32.604091,-85.512268,ArcGIS,8444ee7ffffffff
2,US|AL|394388,"Birmingham, AL",AL,"Birmingham, AL, AL, USA",33.492037,-86.864467,ArcGIS,8444e83ffffffff
3,US|AL|394519,"Daphne, AL",AL,"Daphne, AL, AL, USA",30.869567,-87.776489,ArcGIS,8444507ffffffff
4,US|AL|394537,"Dothan, AL",AL,"Dothan, AL, AL, USA",31.239375,-85.405742,ArcGIS,8444e15ffffffff


In [8]:
# Join geospatial attributes back to regional monthly panel
df_geo = df.merge(
    region_geo[["unique_id", "lat", "lon", "h3_cell"]],
    on="unique_id",
    how="left")

region_df = df_geo[df_geo["hierarchy_level"] == 2].copy()

print("Metropolitan monthly rows:", len(region_df))
print("Rows with H3 cell:", region_df["h3_cell"].notna().sum())
display(region_df.head())

Metropolitan monthly rows: 65400
Rows with H3 cell: 65400


,unique_id,ds,y,RegionName,StateName,hierarchy_level,lat,lon,h3_cell
10900,US|AK|394327,2008-02-29,438.0,"Anchorage, AK",AK,2,61.216583,-149.899597,840c733ffffffff
10901,US|AK|394327,2008-03-31,482.0,"Anchorage, AK",AK,2,61.216583,-149.899597,840c733ffffffff
10902,US|AK|394327,2008-04-30,578.0,"Anchorage, AK",AK,2,61.216583,-149.899597,840c733ffffffff
10903,US|AK|394327,2008-05-31,643.0,"Anchorage, AK",AK,2,61.216583,-149.899597,840c733ffffffff
10904,US|AK|394327,2008-06-30,726.0,"Anchorage, AK",AK,2,61.216583,-149.899597,840c733ffffffff


## Sales aggregation by spatial cell

Sales counts are aggregated by H3 cell to identify areas with higher market activity. The aggregation uses total sales, average monthly sales, and the number of metropolitan markets within each cell.

In [9]:
# Aggregate sales by H3 cell
h3_sales = (
    region_df
    .dropna(subset=["h3_cell"])
    .groupby("h3_cell", as_index=False)
    .agg(
        total_sales=("y", "sum"),
        avg_monthly_sales=("y", "mean"),
        n_regions=("unique_id", "nunique"),
        lat=("lat", "mean"),
        lon=("lon", "mean")
    )
    .sort_values("total_sales", ascending=False)
    .reset_index(drop=True)
)

h3_sales["sales_share"] = h3_sales["total_sales"] / h3_sales["total_sales"].sum()

display(h3_sales.head(20))

,h3_cell,total_sales,avg_monthly_sales,n_regions,lat,lon,sales_share
0,842a107ffffffff,3132864.0,4790.311927,3,40.688226,-73.958030,0.049027
1,842664dffffffff,2189481.0,10043.490826,1,41.936000,-87.740300,0.034264
2,8444a11ffffffff,1944096.0,8917.871560,1,26.096851,-80.134647,0.030424
3,8429a57ffffffff,1826224.0,8377.174312,1,33.998650,-118.256130,0.028579
4,8444c1bffffffff,1786885.0,8196.720183,1,33.778856,-84.385022,0.027964
5,8429b6dffffffff,1623086.0,7445.348624,1,33.448204,-112.072585,0.025400
6,842aa87ffffffff,1512765.0,6939.288991,1,38.805780,-77.152280,0.023674
7,842a135ffffffff,1427157.0,6546.591743,1,40.115991,-75.032100,0.022334
8,8426cb9ffffffff,1425284.0,6538.000000,1,32.777977,-96.796215,0.022305
9,842ab2dffffffff,1284544.0,5892.403670,1,42.353289,-83.085053,0.020102


In [10]:
# Convert H3 cells to hexagon polygons
def h3_to_polygon(cell):
    boundary = h3.cell_to_boundary(cell)
    return Polygon([(lng, lat) for lat, lng in boundary])


h3_sales["geometry"] = h3_sales["h3_cell"].apply(h3_to_polygon)

h3_gdf = gpd.GeoDataFrame(
    h3_sales,
    geometry="geometry",
    crs="EPSG:4326"
)

display(h3_gdf.head())

,h3_cell,total_sales,avg_monthly_sales,n_regions,lat,lon,sales_share,geometry
0,842a107ffffffff,3132864.0,4790.311927,3,40.688226,-73.958030,0.049027,"POLYGON ((-74.09511 40.80687, -74.40767 40.754..."
1,842664dffffffff,2189481.0,10043.490826,1,41.936000,-87.740300,0.034264,"POLYGON ((-87.52625 42.15292, -87.82355 42.064..."
2,8444a11ffffffff,1944096.0,8917.871560,1,26.096851,-80.134647,0.030424,"POLYGON ((-80.11978 25.66954, -79.90854 25.770..."
3,8429a57ffffffff,1826224.0,8377.174312,1,33.998650,-118.256130,0.028579,"POLYGON ((-118.20278 33.53408, -117.97987 33.7..."
4,8444c1bffffffff,1786885.0,8196.720183,1,33.778856,-84.385022,0.027964,"POLYGON ((-84.54957 33.46067, -84.31343 33.559..."


In [11]:
# Regional totals for point overlay
region_totals = (
    region_df
    .dropna(subset=["lat", "lon"])
    .groupby(["unique_id", "RegionName", "StateName", "lat", "lon"], as_index=False)
    .agg(total_sales=("y", "sum"))
)

region_totals["total_sales"] = region_totals["total_sales"].round(0)

region_totals["radius"] = 3 + 10 * np.sqrt(
    region_totals["total_sales"] / region_totals["total_sales"].max()
)

display(region_totals.head())

,unique_id,RegionName,StateName,lat,lon,total_sales,radius
0,US|AK|394327,"Anchorage, AK",AK,61.216583,-149.899597,140865.0,5.182527
1,US|AL|394351,"Auburn, AL",AL,32.604091,-85.512268,24250.0,3.905553
2,US|AL|394388,"Birmingham, AL",AL,33.492037,-86.864467,351440.0,6.447336
3,US|AL|394519,"Daphne, AL",AL,30.869567,-87.776489,91387.0,4.757925
4,US|AL|394537,"Dothan, AL",AL,31.239375,-85.405742,29169.0,3.993159


## Interactive spatial maps

The maps show sales concentration and market density across H3 cells. These visualizations are intended for exploratory analysis, not as exact geographic boundaries of Zillow markets.

In [12]:
# Interactive H3 map: total sales concentration 2008-02-29 to 2026-03-31
os.makedirs("Outputs", exist_ok=True)

h3_map = h3_gdf[h3_gdf["total_sales"] > 0].copy().to_crs(epsg=4326)

h3_map["total_sales"] = h3_map["total_sales"].round(0)
h3_map["avg_monthly_sales"] = h3_map["avg_monthly_sales"].round(2)
h3_map["log_total_sales"] = np.log10(h3_map["total_sales"])

sales_cmap = cm.linear.YlOrRd_09.scale(
    h3_map["log_total_sales"].min(),
    h3_map["log_total_sales"].max()
)

sales_cmap.caption = "log10(total sales count)"

m_sales = folium.Map(
    location=[39.5, -98.35],
    zoom_start=4,
    tiles="CartoDB positron"
)

folium.GeoJson(
    h3_map.to_json(),
    name="H3 sales concentration",
    style_function=lambda feature: {
        "fillColor": sales_cmap(feature["properties"]["log_total_sales"]),
        "color": "black",
        "weight": 0.4,
        "fillOpacity": 0.75
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["h3_cell", "total_sales", "avg_monthly_sales", "n_regions"],
        aliases=["H3 cell", "Total sales", "Avg. monthly sales", "Markets"],
        localize=True,
        sticky=True
    )
).add_to(m_sales)

sales_cmap.add_to(m_sales)
folium.LayerControl().add_to(m_sales)

m_sales.save("Outputs/zillow_h3_sales_concentration.html")

m_sales

In [13]:
# Interactive H3 map: number of regional markets per cell
h3_density = h3_gdf[h3_gdf["n_regions"] > 0].copy().to_crs(epsg=4326)

density_cmap = cm.linear.Blues_09.scale(
    h3_density["n_regions"].min(),
    h3_density["n_regions"].max()
)

density_cmap.caption = "Number of metropolitan markets"

m_density = folium.Map(
    location=[39.5, -98.35],
    zoom_start=4,
    tiles="CartoDB positron"
)

folium.GeoJson(
    h3_density.to_json(),
    name="Metropolitan market density",
    style_function=lambda feature: {
        "fillColor": density_cmap(feature["properties"]["n_regions"]),
        "color": "black",
        "weight": 0.4,
        "fillOpacity": 0.75
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["h3_cell", "n_regions", "total_sales", "avg_monthly_sales"],
        aliases=["H3 cell", "Markets", "Total sales", "Avg. monthly sales"],
        localize=True,
        sticky=True
    )
).add_to(m_density)

density_cmap.add_to(m_density)
folium.LayerControl().add_to(m_density)

m_density.save("Outputs/zillow_h3_market_density.html")

m_density

In [14]:
# Combined interactive map: H3 cells + regional market points
m_combo = folium.Map(
    location=[39.5, -98.35], #US
    zoom_start=4,
    tiles="CartoDB positron")

folium.GeoJson(
    h3_map.to_json(),
    name="H3 sales concentration",
    style_function=lambda feature: {
        "fillColor": sales_cmap(feature["properties"]["log_total_sales"]),
        "color": "black",
        "weight": 0.35,
        "fillOpacity": 0.65
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["h3_cell", "total_sales", "n_regions"],
        aliases=["H3 cell", "Total sales", "Markets"],
        localize=True,
        sticky=True
    )
).add_to(m_combo)

for _, row in region_totals.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=float(row["radius"]),
        color="black",
        weight=0.5,
        fill=True,
        fill_opacity=0.75,
        popup=folium.Popup(
            f"<b>{row['RegionName']}</b><br>"
            f"State group: {row['StateName']}<br>"
            f"Total sales: {row['total_sales']:,.0f}",
            max_width=250
        ),
        tooltip=f"{row['RegionName']} | {row['total_sales']:,.0f}"
    ).add_to(m_combo)

sales_cmap.add_to(m_combo)
folium.LayerControl().add_to(m_combo)

m_combo.save("Outputs/zillow_h3_combined_map.html")

m_combo